In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

df = pd.read_excel("datos_VI1.xlsx")
df["Treatment"] = df["Treatment"].str.strip()
df.head()

,Mice,Treatment,log UFC/g,Blood Counts,BAL LDH,BAL Albumin,BAL IgA,BAL IgG,Serum IgM,Serum IgG,Serum IgA,BAL TNF-a,BAL IFN-g,BAL IL-4,Serum TNF-a,Serum IFN-g,Serum IL-4
0,1,Control,7.84,Positive,123.4,2.1,2.31,3.12,37.81,87.43,15.61,41.6,51.5,46.3,256.2,236.1,406.3
1,2,Control,7.66,Positive,133.4,2.3,2.24,3.11,36.42,88.24,14.22,40.3,50.4,45.7,253.4,237.4,431.4
2,3,Control,7.21,Positive,126.7,1.9,2.01,2.89,37.02,88.01,14.32,41.2,50.3,45.9,255.1,231.2,422.3
3,4,Control,8.13,Positive,125.8,1.9,2.45,3.54,36.87,87.45,16.22,40.9,51.3,46.2,261.3,232.1,412.8
4,5,Control,8.21,Positive,128.4,2.1,2.08,3.49,37.15,88.08,NaN,42.1,51.9,47.4,250.2,233.3,NaN


In [2]:
df["Grupo"] = df["Treatment"].replace({
    "Control": "Control",
    "CP010401": "Cp090104",
    "BPCD": "PCp090104"
})
df["Grupo"].value_counts()

Grupo
Control      6
Cp090104     6
PCp090104    6
Name: count, dtype: int64

In [3]:
df.groupby("Grupo")["Serum IgA"].agg(["mean", "std", "count"])

,mean,std,count
Grupo,,,
Control,15.226000,0.901876,5
Cp090104,20.593333,0.887078,6
PCp090104,18.118000,1.452711,5


In [4]:
grupos = [g["Serum IgA"].dropna() for nombre, g in df.groupby("Grupo")]
f_stat, p_valor = stats.f_oneway(*grupos)
print(f"F = {f_stat:.3f}, p = {p_valor:.4f}")

F = 32.677, p = 0.0000


In [6]:
datos_sin_nan = df.dropna(subset=["Serum IgA"])
tukey = pairwise_tukeyhsd(endog=datos_sin_nan["Serum IgA"], groups=datos_sin_nan["Grupo"], alpha=0.05)
print(tukey)

   Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1    group2  meandiff p-adj   lower   upper  reject
---------------------------------------------------------
 Control  Cp090104   5.3673    0.0  3.6142  7.1205   True
 Control PCp090104    2.892 0.0029  1.0609  4.7231   True
Cp090104 PCp090104  -2.4753 0.0067 -4.2285 -0.7222   True
---------------------------------------------------------


In [7]:
def anova_tukey(variable):
    datos = df.dropna(subset=[variable])
    grupos = [g[variable] for nombre, g in datos.groupby("Grupo")]
    f_stat, p_valor = stats.f_oneway(*grupos)
    print(f"\n--- {variable} ---")
    print(f"F = {f_stat:.3f}, p = {p_valor:.4f}")
    tukey = pairwise_tukeyhsd(endog=datos[variable], groups=datos["Grupo"], alpha=0.05)
    print(tukey)

In [8]:
for var in ["Serum IgA", "Serum IgG", "Serum IgM"]:
    anova_tukey(var)


--- Serum IgA ---
F = 32.677, p = 0.0000
   Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1    group2  meandiff p-adj   lower   upper  reject
---------------------------------------------------------
 Control  Cp090104   5.3673    0.0  3.6142  7.1205   True
 Control PCp090104    2.892 0.0029  1.0609  4.7231   True
Cp090104 PCp090104  -2.4753 0.0067 -4.2285 -0.7222   True
---------------------------------------------------------

--- Serum IgG ---
F = 97.644, p = 0.0000
   Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1    group2  meandiff p-adj   lower   upper  reject
---------------------------------------------------------
 Control  Cp090104   7.7383 0.0435  0.2124 15.2643   True
 Control PCp090104  38.2883    0.0 30.7624 45.8143   True
Cp090104 PCp090104    30.55    0.0  23.024  38.076   True
---------------------------------------------------------

--- Serum IgM ---
F = 735.783, p = 0.0000
  Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 gro

In [14]:
### Nota de auditoría
"""El texto de la tesis (revisado por un tercero) describe una excepción en 
IgM que este análisis, corrido sobre los datos originales, no reproduce: 
las tres inmunoglobulinas séricas muestran diferencia significativa entre 
Cp090104 y PCp090104 (p < 0.001 en los tres casos). Se prioriza el 
resultado del análisis propio sobre la descripción textual."""

'El texto de la tesis (revisado por un tercero) describe una excepción en \nIgM que este análisis, corrido sobre los datos originales, no reproduce: \nlas tres inmunoglobulinas séricas muestran diferencia significativa entre \nCp090104 y PCp090104 (p < 0.001 en los tres casos). Se prioriza el \nresultado del análisis propio sobre la descripción textual.'

In [15]:
citoquinas_bal = df.melt(
    id_vars=["Grupo"],
    value_vars=["BAL TNF-a", "BAL IFN-g", "BAL IL-4"],
    var_name="Citoquina",
    value_name="Valor"
)
citoquinas_bal.head()

,Grupo,Citoquina,Valor
0,Control,BAL TNF-a,41.6
1,Control,BAL TNF-a,40.3
2,Control,BAL TNF-a,41.2
3,Control,BAL TNF-a,40.9
4,Control,BAL TNF-a,42.1


In [16]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

modelo = ols('Valor ~ C(Grupo) * C(Citoquina)', data=citoquinas_bal).fit()
tabla_anova = sm.stats.anova_lm(modelo, typ=2)
print(tabla_anova)

                            sum_sq    df            F        PR(>F)
C(Grupo)                635.921893   2.0   367.134538  1.362430e-28
C(Citoquina)           5182.513226   2.0  2992.002042  1.385977e-48
C(Grupo):C(Citoquina)  1244.818452   4.0   359.333318  1.619040e-33
Residual                 38.972750  45.0          NaN           NaN


In [17]:
for var in ["BAL TNF-a", "BAL IFN-g", "BAL IL-4"]:
    anova_tukey(var)


--- BAL TNF-a ---
F = 77.026, p = 0.0000
   Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1    group2  meandiff p-adj   lower   upper  reject
---------------------------------------------------------
 Control  Cp090104     -7.9    0.0 -9.5726 -6.2274   True
 Control PCp090104     -5.0    0.0 -6.6726 -3.3274   True
Cp090104 PCp090104      2.9 0.0012  1.2274  4.5726   True
---------------------------------------------------------

--- BAL IFN-g ---
F = 823.537, p = 0.0000
   Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1    group2  meandiff p-adj  lower    upper  reject
---------------------------------------------------------
 Control  Cp090104  16.8167   0.0  15.7336 17.8998   True
 Control PCp090104   6.7683   0.0   5.6852  7.8514   True
Cp090104 PCp090104 -10.0483   0.0 -11.1314 -8.9652   True
---------------------------------------------------------

--- BAL IL-4 ---
F = 498.430, p = 0.0000
  Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 gro

In [18]:
for var in ["Serum TNF-a", "Serum IFN-g", "Serum IL-4"]:
    anova_tukey(var)


--- Serum TNF-a ---
F = 1524.758, p = 0.0000
   Multiple Comparison of Means - Tukey HSD, FWER=0.05    
 group1    group2  meandiff p-adj  lower    upper   reject
----------------------------------------------------------
 Control  Cp090104 -72.5667   0.0 -76.1575 -68.9758   True
 Control PCp090104 -56.8167   0.0 -60.4075 -53.2258   True
Cp090104 PCp090104    15.75   0.0  12.1591  19.3409   True
----------------------------------------------------------

--- Serum IFN-g ---
F = 188.213, p = 0.0000
   Multiple Comparison of Means - Tukey HSD, FWER=0.05    
 group1    group2  meandiff p-adj   lower    upper  reject
----------------------------------------------------------
 Control  Cp090104     75.3    0.0  64.8316 85.7684   True
 Control PCp090104     55.9    0.0  45.4316 66.3684   True
Cp090104 PCp090104    -19.4 0.0006 -29.8684 -8.9316   True
----------------------------------------------------------

--- Serum IL-4 ---
F = 27.117, p = 0.0001
    Multiple Comparison of Means - Tukey

In [19]:
citoquinas_todas = df.melt(
    id_vars=["Grupo"],
    value_vars=["BAL TNF-a", "BAL IFN-g", "BAL IL-4", "Serum TNF-a", "Serum IFN-g", "Serum IL-4"],
    var_name="Variable",
    value_name="Valor"
)

# separar compartimento (BAL/Serum) y citoquina en dos columnas
citoquinas_todas[["Compartimento", "Citoquina"]] = citoquinas_todas["Variable"].str.split(" ", n=1, expand=True)

resumen = citoquinas_todas.groupby(["Compartimento", "Citoquina", "Grupo"])["Valor"].agg(["mean", "std"]).reset_index()
resumen

,Compartimento,Citoquina,Grupo,mean,std
0,BAL,IFN-g,Control,50.950000,0.703562
1,BAL,IFN-g,Cp090104,67.766667,0.784007
2,BAL,IFN-g,PCp090104,57.718333,0.674697
3,BAL,IL-4,Control,46.200000,0.638749
4,BAL,IL-4,Cp090104,62.500000,0.672309
5,BAL,IL-4,PCp090104,57.216667,1.279714
6,BAL,TNF-a,Control,40.800000,1.196662
7,BAL,TNF-a,Cp090104,32.900000,1.214907
8,BAL,TNF-a,PCp090104,35.800000,0.907744
9,Serum,IFN-g,Control,234.033333,2.362767


import matplotlib.pyplot as plt
import numpy as np

colores = {"Control": "#52514e", "Cp090104": "#2a78d6", "PCp090104": "#eb6834"}
citoquinas = ["TNF-a", "IFN-g", "IL-4"]
grupos = ["Control", "Cp090104", "PCp090104"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False)

for ax, compartimento in zip(axes, ["BAL", "Serum"]):
    x = np.arange(len(citoquinas))
    ancho = 0.25
    for i, grupo in enumerate(grupos):
        sub = resumen[(resumen["Compartimento"] == compartimento) & (resumen["Grupo"] == grupo)]
        sub = sub.set_index("Citoquina").loc[citoquinas]
        ax.bar(x + i*ancho, sub["mean"], ancho, yerr=sub["std"], capsize=3,
               label=grupo, color=colores[grupo])
    ax.set_xticks(x + ancho)
    ax.set_xticklabels(citoquinas)
    ax.set_title(f"{compartimento}")
    ax.set_ylabel("pg/mL")

axes[0].legend()
fig.suptitle("Citoquinas en BAL y suero — día 7 post-infección", y=1.03)
plt.tight_layout()
plt.savefig("citoquinas_BAL_suero.png", dpi=150, bbox_inches="tight")
plt.show()

## Conclusión

**Pregunta:** ¿el tratamiento nasal con Cp090104 y su BLP modifica los niveles de
TNF-α, IFN-γ e IL-4 en tracto respiratorio (BAL) y en suero, al día 7 post-infección?

**Lo que decía el texto original:** las tres citoquinas suben en BAL; en suero
suben IFN-γ e IL-4, pero el TNF-α baja.

**Lo que muestra este análisis sobre el dato original** (ANOVA de una vía + Tukey,
n = 6/grupo, Python): IFN-γ e IL-4 suben significativamente tanto en BAL como en
suero. El **TNF-α baja significativamente en los dos compartimentos**, no solo en
suero — un matiz que el resumen original no reportaba para BAL.

**Un segundo hallazgo, sobre Serum IgM:** el resumen original decía que la
bacteria viva y su derivado no viable (BLP) no se diferenciaban entre sí en este
isotipo. El análisis propio, sobre el dato original, muestra una diferencia
significativa entre ambos en los tres pares de grupos comparados.

**Conclusión metodológica:** trabajar sobre el dato original, en vez de aceptar
un resumen ya escrito, permitió encontrar dos matices que ese resumen no
reflejaba con precisión — y es, en definitiva, la habilidad que este ejercicio
buscaba practicar: auditar un resultado en vez de darlo por bueno.